# Setup

In [17]:
import os

import numpy as np
import pandas as pd

from data_loader import fetch_nasa_power, load_hdx_rainfall, load_faostat_syria

# Load Data

In [ ]:
syria_regions = { # lat, lon
    "Hassakeh": (36.5, 40.75),
    "Aleppo": (36.2, 37.17),
    "Idleb": (35.93, 36.63),
    "Raqqa": (35.95, 39.0),
    "Homs": (34.73, 36.72),
}

# Fetch data for each region
nasa_data = {}

for region, (lat, lon) in syria_regions.items():
    print(f"\nFetching NASA POWER data for {region} ({lat}, {lon})...")
    df = fetch_nasa_power(region, lat, lon)
    if df is not None:
        nasa_data[region] = df
        print(f"  Retrieved {len(df)} daily records ({df.index.min().date()} to {df.index.max().date()})")
    else:
        print(f"  Failed to retrieve data for {region}")


Fetching NASA POWER data for Hassakeh (36.5, 40.75)...
  [DOWNLOAD] Fetching from NASA POWER API for Hassakeh...
  [SAVED] ./SyriaWheatWatch\data_raw\nasa_daily_hassakeh.csv (12784 rows)
  Retrieved 12784 daily records (1991-01-01 to 2025-12-31)

Fetching NASA POWER data for Aleppo (36.2, 37.17)...
  [DOWNLOAD] Fetching from NASA POWER API for Aleppo...
  [SAVED] ./SyriaWheatWatch\data_raw\nasa_daily_aleppo.csv (12784 rows)
  Retrieved 12784 daily records (1991-01-01 to 2025-12-31)

Fetching NASA POWER data for Idleb (35.93, 36.63)...
  [DOWNLOAD] Fetching from NASA POWER API for Idleb...
  [SAVED] ./SyriaWheatWatch\data_raw\nasa_daily_idleb.csv (12784 rows)
  Retrieved 12784 daily records (1991-01-01 to 2025-12-31)

Fetching NASA POWER data for Raqqa (35.95, 39.0)...
  [DOWNLOAD] Fetching from NASA POWER API for Raqqa...
  [SAVED] ./SyriaWheatWatch\data_raw\nasa_daily_raqqa.csv (12784 rows)
  Retrieved 12784 daily records (1991-01-01 to 2025-12-31)

Fetching NASA POWER data for Homs 

In [19]:
rainfall_df = load_hdx_rainfall()

  [CACHE] Loading from Drive: ./SyriaWheatWatch\data_raw\syr-rainfall-subnat-full.csv
  [SAVED] Clean version: ./SyriaWheatWatch\data_raw\syr-rainfall-subnat-full.csv


In [20]:
wheat_df = load_faostat_syria()

  [CACHE] Loading from Drive: ./SyriaWheatWatch\data_raw\faostat_syria_crops.csv


# Preprocess

In [43]:
region_gs = {}

for region, df in nasa_data.items():
    temp = df.copy()
    temp["year"]  = temp.index.year
    temp["month"] = temp.index.month

    # Filter to growing-season months
    gs = temp[temp["month"].isin([11, 12, 1, 2, 3, 4, 5])].copy()

    # Nov–Dec belong to the NEXT year's growing season
    gs["gs_year"] = gs["year"]
    gs.loc[gs["month"].isin([11, 12]), "gs_year"] = gs.loc[gs["month"].isin([11, 12]), "year"] + 1

    region_gs[region] = gs.groupby("gs_year")["solar_kwh_m2"].mean()

solar_gs_by_region = pd.DataFrame(region_gs)
solar_clean = (
    solar_gs_by_region
    .mean(axis=1) 
    .rename("growing_season_solar_kwh_m2") 
    .reset_index() 
    .rename(columns={"gs_year": "year"})
)
solar_clean.head()

,year,growing_season_solar_kwh_m2
0,1991,15.756980
1,1992,14.629812
2,1993,14.285094
3,1994,15.023151
4,1995,14.497783


In [44]:
wheat_clean = (
    wheat_df
    .pivot_table(
        index="Year",
        columns="Element",
        values="Value",
        aggfunc="sum",
    )
    .reset_index()
    .rename(columns={
        "Year":            "year",
        "Production":      "wheat_production_t",
        "Area harvested":  "wheat_area_ha",
        "Yield":           "wheat_yield_kg_ha",
    })
    .sort_values("year")
    .reset_index(drop=True)
)

wheat_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                64 non-null     int64  
 1   wheat_area_ha       64 non-null     float64
 2   wheat_production_t  64 non-null     float64
 3   wheat_yield_kg_ha   64 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 2.1 KB


In [46]:
import pandas as pd

# Ensure date is datetime
rainfall_df["date"] = pd.to_datetime(rainfall_df["date"])

# Add year and month
rainfall_df["year"]  = rainfall_df["date"].dt.year
rainfall_df["month"] = rainfall_df["date"].dt.month

# Growing season months (Nov–May)
gs_months = [11, 12, 1, 2, 3, 4, 5]

# --- Growing season total (Nov–May, sum) ---
gs = rainfall_df[rainfall_df["month"].isin(gs_months)].copy()
gs["gs_year"] = gs["year"]
gs.loc[gs["month"].isin([11, 12]), "gs_year"] = gs.loc[gs["month"].isin([11, 12]), "year"] + 1

rainfall_clean = (
    gs
    .groupby("gs_year", as_index=False)["rfh"]
    .sum()
    .rename(columns={"gs_year": "year", "rfh": "growing_season_rainfall_mm"})
)

rainfall_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 2 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        46 non-null     int32  
 1   growing_season_rainfall_mm  46 non-null     float64
dtypes: float64(1), int32(1)
memory usage: 680.0 bytes


# Merge

In [47]:
merged = (
    solar_clean
    .merge(rainfall_clean, on="year", how="inner")
    .merge(wheat_clean, on="year", how="inner")
    .sort_values("year")
    .reset_index(drop=True)
)

print(f"Merged shape: {merged.shape}")
print(f"Years: {merged['year'].min()}–{merged['year'].max()}")
print(f"Columns: {merged.columns.tolist()}")
merged.head()

Merged shape: (34, 6)
Years: 1991–2024
Columns: ['year', 'growing_season_solar_kwh_m2', 'growing_season_rainfall_mm', 'wheat_area_ha', 'wheat_production_t', 'wheat_yield_kg_ha']


,year,growing_season_solar_kwh_m2,growing_season_rainfall_mm,wheat_area_ha,wheat_production_t,wheat_yield_kg_ha
0,1991,15.756980,27711.529927,3502000.0,3057000.0,2097.1
1,1992,14.629812,30559.497185,3648000.0,4137400.0,2687.0
2,1993,14.285094,28024.360447,3554011.0,5179843.0,3334.4
3,1994,15.023151,25651.197564,3447409.0,5184725.0,3166.1
4,1995,14.497783,26002.788335,3606861.0,5889286.0,3414.2


# Save

In [49]:
from  config import DATA_CLEAN

In [50]:
out = os.path.join(DATA_CLEAN, "yearly_clean.csv")
merged.to_csv(out, index=False)